In [1]:
import sklearn

import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Load your data
df = pd.read_csv("../combined_dataset.csv")
df_both = df


In [3]:
df

,Unnamed: 0,eid,split,X0,X1,X2,X3,X4,X5,X6,...,X70,status,sex,LVEDV,LVESV,RVEDV,RVESV,LVepiEDV,LVEF,RVEF
0,1,1013662,test,308.624395,435.034217,205.582358,28.305618,55.140791,1079.404231,56.236569,...,-7.195731,pMI,1,184753.610587,76584.049900,207969.748253,93296.049282,314036.262430,0.585480,0.551396
1,2,1020505,train,199.011579,616.384370,2.628365,75.606711,149.211427,1149.665670,-345.077163,...,33.771369,healthy,1,142030.786211,65000.318139,169494.131694,83292.186396,279656.061727,0.542351,0.508584
2,3,1028018,test,2679.759925,-6.335275,90.473636,-274.517816,-106.285329,-645.708287,53.634358,...,-7.917882,pMI,1,174880.492409,105680.587691,212389.503012,129881.202772,297549.912601,0.395698,0.388476
3,4,1031970,test,-3624.258305,-9.516214,-48.723806,-95.552690,70.909980,-1335.315075,-38.076555,...,24.318811,pMI,0,118366.752157,47554.063072,131923.294966,58864.966152,225823.457925,0.598248,0.553794
4,5,1036633,train,871.477817,-24.350968,-545.529805,212.756967,60.365124,603.189524,-29.175393,...,3.630067,healthy,1,152758.045966,56483.263466,169902.304325,75815.766555,267780.410158,0.630244,0.553768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,828,5523416,train,5480.311519,550.036318,-117.048572,166.942504,387.304188,-540.517023,24.668981,...,45.529308,healthy,1,134841.545775,46023.447967,141917.424496,55797.839572,220481.736330,0.658685,0.606829
828,829,5655739,train,-448.908272,384.098326,816.726705,-379.742997,126.471366,1995.327352,13.248153,...,-33.871382,healthy,0,162845.549082,70798.732443,170311.921619,74297.644720,274144.411293,0.565240,0.563755
829,830,5766740,valid,-988.557139,-749.386333,8.941719,-168.663784,-248.855457,-3965.100957,111.888417,...,-14.222355,healthy,1,139861.943618,48522.885402,159394.507112,64019.753939,235820.166493,0.653066,0.598357
830,831,5886788,train,4447.378266,-82.756710,-225.425852,20.370201,202.330860,-3749.470788,163.900498,...,-45.969860,healthy,1,121252.150488,56954.188912,139773.118062,63970.917866,247910.215443,0.530283,0.542323


In [76]:
# 1. Filter for 'healthy' and 'iMI' only
df_filtered = df_both[df_both["status"].isin(["healthy", "iMI"])]

# 2. Features and labels
X = df_filtered[["LVEF", "RVEF"]]
y = df_filtered["status"].map({"healthy": 0, "iMI": 1})  # ✅ Convert to binary
sex = df_filtered["sex"]

# 3. Split by sex
X_female = X[sex == 0]
y_female = y[sex == 0]
X_male = X[sex == 1]
y_male = y[sex == 1]

# 4. Define model pipeline
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear")  # L1 optional
)

# 5. Define metrics and CV
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6. Cross-validation
results_female = cross_validate(pipeline, X_female, y_female, scoring=scoring, cv=cv)
results_male = cross_validate(pipeline, X_male, y_male, scoring=scoring, cv=cv)

# 7. Display results
def print_results(label, results):
    print(f"\n📊 {label} Performance:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"{metric.upper():<10}: {scores.mean():.3f} ± {scores.std():.3f}")

print_results("Female", results_female)
print_results("Male", results_male)


📊 Female Performance:
ACCURACY  : 0.899 ± 0.006
PRECISION : 0.000 ± 0.000
RECALL    : 0.000 ± 0.000
F1        : 0.000 ± 0.000
ROC_AUC   : 0.415 ± 0.097

📊 Male Performance:
ACCURACY  : 0.744 ± 0.023
PRECISION : 0.645 ± 0.154
RECALL    : 0.205 ± 0.041
F1        : 0.309 ± 0.060
ROC_AUC   : 0.673 ± 0.085


/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

In [77]:
df

,Unnamed: 0,eid,split,X0,X1,X2,X3,X4,X5,X6,...,X70,status,sex,LVEDV,LVESV,RVEDV,RVESV,LVepiEDV,LVEF,RVEF
0,1,1013662,test,308.624395,435.034217,205.582358,28.305618,55.140791,1079.404231,56.236569,...,-7.195731,pMI,1,184753.610587,76584.049900,207969.748253,93296.049282,314036.262430,0.585480,0.551396
1,2,1020505,train,199.011579,616.384370,2.628365,75.606711,149.211427,1149.665670,-345.077163,...,33.771369,healthy,1,142030.786211,65000.318139,169494.131694,83292.186396,279656.061727,0.542351,0.508584
2,3,1028018,test,2679.759925,-6.335275,90.473636,-274.517816,-106.285329,-645.708287,53.634358,...,-7.917882,pMI,1,174880.492409,105680.587691,212389.503012,129881.202772,297549.912601,0.395698,0.388476
3,4,1031970,test,-3624.258305,-9.516214,-48.723806,-95.552690,70.909980,-1335.315075,-38.076555,...,24.318811,pMI,0,118366.752157,47554.063072,131923.294966,58864.966152,225823.457925,0.598248,0.553794
4,5,1036633,train,871.477817,-24.350968,-545.529805,212.756967,60.365124,603.189524,-29.175393,...,3.630067,healthy,1,152758.045966,56483.263466,169902.304325,75815.766555,267780.410158,0.630244,0.553768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,828,5523416,train,5480.311519,550.036318,-117.048572,166.942504,387.304188,-540.517023,24.668981,...,45.529308,healthy,1,134841.545775,46023.447967,141917.424496,55797.839572,220481.736330,0.658685,0.606829
828,829,5655739,train,-448.908272,384.098326,816.726705,-379.742997,126.471366,1995.327352,13.248153,...,-33.871382,healthy,0,162845.549082,70798.732443,170311.921619,74297.644720,274144.411293,0.565240,0.563755
829,830,5766740,valid,-988.557139,-749.386333,8.941719,-168.663784,-248.855457,-3965.100957,111.888417,...,-14.222355,healthy,1,139861.943618,48522.885402,159394.507112,64019.753939,235820.166493,0.653066,0.598357
830,831,5886788,train,4447.378266,-82.756710,-225.425852,20.370201,202.330860,-3749.470788,163.900498,...,-45.969860,healthy,1,121252.150488,56954.188912,139773.118062,63970.917866,247910.215443,0.530283,0.542323


In [78]:
# 1. Filter for 'healthy' and 'iMI' only
df_filtered = df_both[df_both["status"].isin(["healthy", "iMI"])]
df_filtered = df_filtered[df_both["split"].isin(["test", "valid"])]

# 2. Features and labels
X = df_filtered[["LVEF", "RVEF"]]
y = df_filtered["status"].map({"healthy": 0, "iMI": 1})  # ✅ Convert to binary
sex = df_filtered["sex"]

# 3. Split by sex
X_female = X[sex == 0]
y_female = y[sex == 0]
X_male = X[sex == 1]
y_male = y[sex == 1]

# 4. Define model pipeline
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear")  # L1 optional
)

# 5. Define metrics and CV
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6. Cross-validation
results_female = cross_validate(pipeline, X_female, y_female, scoring=scoring, cv=cv)
results_male = cross_validate(pipeline, X_male, y_male, scoring=scoring, cv=cv)

# 7. Display results
def print_results(label, results):
    print(f"\n📊 {label} Performance:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"{metric.upper():<10}: {scores.mean():.3f} ± {scores.std():.3f}")

print_results("Female", results_female)
print_results("Male", results_male)


📊 Female Performance:
ACCURACY  : 0.649 ± 0.023
PRECISION : 0.000 ± 0.000
RECALL    : 0.000 ± 0.000
F1        : 0.000 ± 0.000
ROC_AUC   : 0.468 ± 0.118

📊 Male Performance:
ACCURACY  : 0.678 ± 0.013
PRECISION : 0.678 ± 0.013
RECALL    : 1.000 ± 0.000
F1        : 0.808 ± 0.009
ROC_AUC   : 0.532 ± 0.068


/tmp/ipykernel_1707073/3881439295.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtered = df_filtered[df_both["split"].isin(["test", "valid"])]
/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wolf6273/miniconda3/envs/4D_geom/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision 

In [79]:
# 1. Filter for 'healthy' and 'iMI' only
df_filtered = df_both[df_both["status"].isin(["healthy", "iMI"])]

# 2. Use all columns that start with 'X' as features
X_cols = [col for col in df_filtered.columns if col.startswith("X")]
X = df_filtered[X_cols]
y = df_filtered["status"].map({"healthy": 0, "iMI": 1})  # Binary label
sex = df_filtered["sex"]

# 3. Split by sex
X_female = X[sex == 0]
y_female = y[sex == 0]
X_male = X[sex == 1]
y_male = y[sex == 1]

# 4. Define model pipeline with L1 regularisation
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", penalty="l1", C=1.0)  # L1 penalty
)

# 5. Define metrics and CV
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6. Cross-validation
results_female = cross_validate(pipeline, X_female, y_female, scoring=scoring, cv=cv)
results_male = cross_validate(pipeline, X_male, y_male, scoring=scoring, cv=cv)

# 7. Display results
def print_results(label, results):
    print(f"\n📊 {label} Performance:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"{metric.upper():<10}: {scores.mean():.3f} ± {scores.std():.3f}")

print_results("Female", results_female)
print_results("Male", results_male)


📊 Female Performance:
ACCURACY  : 0.854 ± 0.023
PRECISION : 0.212 ± 0.165
RECALL    : 0.129 ± 0.065
F1        : 0.149 ± 0.078
ROC_AUC   : 0.649 ± 0.091

📊 Male Performance:
ACCURACY  : 0.675 ± 0.038
PRECISION : 0.393 ± 0.094
RECALL    : 0.301 ± 0.080
F1        : 0.338 ± 0.080
ROC_AUC   : 0.615 ± 0.088


In [80]:
# 1. Filter for 'healthy' and 'iMI' only
df_filtered = df_both[df_both["status"].isin(["healthy", "iMI"])]
df_filtered = df_filtered[df_both["split"].isin(["test", "valid"])]

# 2. Use all columns that start with 'X' as features
X_cols = [col for col in df_filtered.columns if col.startswith("X")]
X = df_filtered[X_cols]
y = df_filtered["status"].map({"healthy": 0, "iMI": 1})  # Binary label
sex = df_filtered["sex"]

# 3. Split by sex
X_female = X[sex == 0]
y_female = y[sex == 0]
X_male = X[sex == 1]
y_male = y[sex == 1]

# 4. Define model pipeline with L1 regularisation
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", penalty="l1", C=1.0)  # L1 penalty
)

# 5. Define metrics and CV
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6. Cross-validation
results_female = cross_validate(pipeline, X_female, y_female, scoring=scoring, cv=cv)
results_male = cross_validate(pipeline, X_male, y_male, scoring=scoring, cv=cv)

# 7. Display results
def print_results(label, results):
    print(f"\n📊 {label} Performance:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"{metric.upper():<10}: {scores.mean():.3f} ± {scores.std():.3f}")

print_results("Female", results_female)
print_results("Male", results_male)

/tmp/ipykernel_1707073/3507835433.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_filtered = df_filtered[df_both["split"].isin(["test", "valid"])]



📊 Female Performance:
ACCURACY  : 0.604 ± 0.096
PRECISION : 0.424 ± 0.126
RECALL    : 0.390 ± 0.175
F1        : 0.391 ± 0.133
ROC_AUC   : 0.736 ± 0.122

📊 Male Performance:
ACCURACY  : 0.587 ± 0.086
PRECISION : 0.695 ± 0.059
RECALL    : 0.702 ± 0.063
F1        : 0.698 ± 0.060
ROC_AUC   : 0.569 ± 0.108
